In [27]:
# Set project root so "from src..." works. Edit the path below if you get ModuleNotFoundError.
import sys
from pathlib import Path
PROJECT_ROOT = Path("/Users/tomosborne/PycharmProjects/PowerPlan")  # <-- edit if your repo is elsewhere
if PROJECT_ROOT.exists():
    sys.path.insert(0, str(PROJECT_ROOT))

# Energy balancing model — Usage

**Inputs:** latitude, longitude (location), solar/wind **capacity** (kW), and **generation-type** params from `src.data.energy_tiers` (budget, mid, premium). **Demand** can be adjusted by **insulation** (R-value) and **heat pump** (COP) before sizing.

**Flows:**
1. **Daily generation** — `get_flux_daily` + `get_generation`: short-range forecast → daily solar/wind kWh.
2. **Last year monthly** — `get_flux_monthly_last_year`: Historical API → 12 months GHI + wind (used for optimisation).
3. **Demand adjustment** — Insulation reduces heating demand; heat pump reduces electricity needed for heating (COP > 1). Use `heating_fraction`, `insulation_r_value`, `heat_pump_cop` in `get_optimised_system` (or `HEAT_PUMP_TIERS`).
4. **Optimisation** — `get_optimised_system(..., flux_source='last_year_monthly')`: sizes and prices system using last year's weather; returns **financials over 0, 5 and 10 years**. Use `flux_source='forecast'` for 7-day forecast instead.
5. **Tariff recommendation** — Use scraped tariff data with `recommend_tariff(tariffs, lat, lon, annual_kwh, ...)` to get a recommended tariff based on optimal solar/wind and each tariff's unit rate and standing charge. Run **ScrapeSequence** and choose "Get tariff recommendation" to use live scraped data.

In [28]:
# Location + capacity + generation types (run project-root cell first)
import importlib
import sys
# Force reload from disk so the kernel sees latest code (fixes stale import errors)
for name in list(sys.modules):
    if name in ("src.models.energy_balancing", "src.api.get_weather", "src.data.energy_tiers") or name.startswith("src.models.energy_balancing."):
        del sys.modules[name]
import src.models.energy_balancing as _eb
importlib.reload(_eb)
get_generation = _eb.get_generation
get_flux_daily = _eb.get_flux_daily
get_flux_monthly_last_year = _eb.get_flux_monthly_last_year
get_optimised_system = _eb.get_optimised_system
DEFAULT_PRICING = _eb.DEFAULT_PRICING
from src.data.energy_tiers import SOLAR_TIERS, WIND_TIERS, HEAT_PUMP_TIERS

# Inputs: location and capacity
latitude = 51.4552   # e.g. Bristol, UK
longitude = -2.5966
solar_capacity_kw = 4.0
wind_capacity_kw = 2.0

# Generation types (stored in src.data.energy_tiers)
solar_type = SOLAR_TIERS["budget"]
wind_type = WIND_TIERS["budget"]

# Optional: date range (default: next 7 days from today)
# start_date, end_date = "2026-02-20", "2026-02-26"

daily = get_generation(
    latitude, longitude,
    solar_capacity_kw, wind_capacity_kw,
    solar_type, wind_type,
)
daily

,ghi_mj_per_m2,wind_speed_10m_max,solar_gen_kwh,wind_gen_kwh,total_gen_kwh
date,,,,,
2026-03-04 00:00:00+00:00,12.39,4.642467,10.611347,1.598636,12.209983
2026-03-05 00:00:00+00:00,11.85,4.348850,10.148867,1.078161,11.227028
2026-03-06 00:00:00+00:00,3.88,5.158004,3.323005,2.759692,6.082697
2026-03-07 00:00:00+00:00,3.59,2.773536,3.074635,0.000000,3.074635
2026-03-08 00:00:00+00:00,4.87,3.063495,4.170884,0.002389,4.173273
2026-03-09 00:00:00+00:00,5.70,3.422353,4.881733,0.105708,4.987441
2026-03-10 00:00:00+00:00,8.50,8.031967,7.279778,15.004855,22.284633


In [29]:
# Summary for the period: total solar, wind, and combined (kWh)
print("Period total — solar:", round(daily["solar_gen_kwh"].sum(), 1), "kWh, wind:", round(daily["wind_gen_kwh"].sum(), 1), "kWh, total:", round(daily["total_gen_kwh"].sum(), 1), "kWh")

Period total — solar: 43.5 kWh, wind: 20.5 kWh, total: 64.0 kWh


In [30]:
# Last year's weather by month (Historical API) — used for optimisation below
try:
    flux_monthly = get_flux_monthly_last_year(latitude, longitude)
    print("Last year monthly flux (GHI = shortwave sum MJ/m², wind = mean max m/s):")
except Exception as e:
    print("Could not load last-year monthly data (Historical API):", e)
    print("Optimisation cell will fall back to flux_source='forecast'.")
    flux_monthly = None
flux_monthly

Last year monthly flux (GHI = shortwave sum MJ/m², wind = mean max m/s):


,ghi_mj_per_m2,wind_speed_10m_max,days_in_month
month,,,
1,100.430000,5.867279,31
2,133.580002,6.174774,28
3,367.750000,4.611729,31
4,520.820007,5.177849,30
5,660.460022,5.332867,31
6,648.210022,5.992554,30
7,654.010010,5.250819,31
8,537.010010,5.361267,31
9,366.770020,5.985127,30


In [31]:
# Optimisation: size and price (last year's monthly weather, or forecast if that fails)
annual_consumption_kwh = 3500  # your annual demand (kWh) before insulation/heat-pump adjustment

# Demand adjustment: insulation reduces heating demand; heat pump reduces electricity for heating (COP)
heating_fraction = 0.6           # share of demand that is space heating (0–1)
insulation_r_value = 0.0         # R-value m²·K/W; 0 = no extra insulation (e.g. 5 for well insulated)
heat_pump_params = HEAT_PUMP_TIERS["standard"]  # "none" (COP 1), "standard" (2.5), "efficient" (3.5)
heat_pump_cop = heat_pump_params["cop"]

try:
    result = get_optimised_system(
        latitude, longitude,
        annual_consumption_kwh,
        solar_type, wind_type,
        solar_max_kw=20.0,
        wind_max_kw=10.0,
        step_kw=0.5,
        optimize_over_years=5.0,
        flux_source="last_year_monthly",
        min_wind_kw=0.5,   # require at least 0.5 kW wind (set to 0 to allow wind=0)
        heating_fraction=heating_fraction,
        insulation_r_value=insulation_r_value,
        heat_pump_cop=heat_pump_cop,
    )
except Exception as e:
    print("Last-year monthly failed, using 7-day forecast instead:", e)
    result = get_optimised_system(
        latitude, longitude, annual_consumption_kwh, solar_type, wind_type,
        solar_max_kw=20.0, wind_max_kw=10.0, step_kw=0.5, optimize_over_years=5.0,
        flux_source="forecast",
        min_wind_kw=0.5,
        heating_fraction=heating_fraction,
        insulation_r_value=insulation_r_value,
        heat_pump_cop=heat_pump_cop,
    )

print("--- Optimal system (sized to minimise cost over 5 years) ---")
print("Flux data:", result["flux_source"], "(period:", result["flux_period_days"], "days)")
# Demand: before adjustments vs after insulation + heat pump (used for sizing)
if result.get("annual_demand_before_adjustments_kwh") is not None:
    print("Demand: before adjustments", result["annual_demand_before_adjustments_kwh"], "kWh  →  after insulation + heat pump (sizing):", result["annual_demand_kwh"], "kWh")
    print("  (heating fraction:", result.get("heating_fraction"), "| insulation R:", result.get("insulation_r_value"), "| heat pump COP:", result.get("heat_pump_cop"), ")")
print("Capacity: solar", result["optimal_solar_kw"], "kW  |  wind", result["optimal_wind_kw"], "kW  (total", result["total_capacity_kw"], "kW)")
print("Annual demand (for balance):", result["annual_demand_kwh"], "kWh  |  Annual generation:", result["annual_generation_kwh"], "kWh")
print("  → solar:", result.get("annual_solar_generation_kwh", 0), "kWh  |  wind:", result.get("annual_wind_generation_kwh", 0), "kWh")
print("Demand met from own generation:", result["demand_met_from_generation_pct"], "%")
print()
print("Capex: solar £" + str(result.get("solar_capex", 0)) + "  |  wind £" + str(result.get("wind_capex", 0)))
print("Payback: solar", result.get("payback_solar_years") or "—", "years  |  wind", result.get("payback_wind_years") or "—", "years")
print()
print("Financials:")
print("  0 year (upfront):  £", result["financials_0_year"]["total"], " (capex only)")
print("  5 year total:      £", result["financials_5_year"]["total"], " (capex + 5 yr opex)")
print("  10 year total:    £", result["financials_10_year"]["total"], " (capex + 10 yr opex)")
print("  (opex = grid import cost − export revenue)")
result

--- Optimal system (sized to minimise cost over 5 years) ---
Flux data: last_year_monthly (period: 365 days)
Demand: before adjustments 3500 kWh  →  after insulation + heat pump (sizing): 2240.0 kWh
  (heating fraction: 0.6 | insulation R: 0.0 | heat pump COP: 2.5 )
Capacity: solar 1.0 kW  |  wind 0.5 kW  (total 1.5 kW)
Annual demand (for balance): 2240.0 kWh  |  Annual generation: 1370.7 kWh
  → solar: 937.3 kWh  |  wind: 433.4 kWh
Demand met from own generation: 61.2 %

Capex: solar £1500.0  |  wind £1250.0
Payback: solar 6.4 years  |  wind 11.5 years

Financials:
  0 year (upfront):  £ 2750.0  (capex only)
  5 year total:      £ 3836.64  (capex + 5 yr opex)
  10 year total:    £ 4923.28  (capex + 10 yr opex)
  (opex = grid import cost − export revenue)


{'optimal_solar_kw': 1.0,
 'optimal_wind_kw': 0.5,
 'total_capacity_kw': 1.5,
 'annual_demand_kwh': 2240.0,
 'annual_generation_kwh': 1370.7,
 'annual_solar_generation_kwh': 937.3,
 'annual_wind_generation_kwh': 433.4,
 'annual_import_kwh': 869.3,
 'annual_export_kwh': 0.0,
 'demand_met_from_generation_pct': 61.2,
 'capex': 2750.0,
 'solar_capex': 1500.0,
 'wind_capex': 1250.0,
 'annual_net_opex': 217.33,
 'payback_solar_years': 6.4,
 'payback_wind_years': 11.5,
 'financials_0_year': {'capex': 2750.0, 'opex_total': 0.0, 'total': 2750.0},
 'financials_5_year': {'capex': 2750.0,
  'opex_total': 1086.64,
  'total': 3836.64},
 'financials_10_year': {'capex': 2750.0,
  'opex_total': 2173.28,
  'total': 4923.28},
 'monthly_balance':    month  solar_kwh  wind_kwh  total_gen_kwh  demand_kwh  import_kwh  \
 0    Jan       21.5      37.8           59.3       186.7       127.4   
 1    Feb       28.6      41.8           70.4       186.7       116.3   
 2    Mar       78.7      11.9           90.7

In [32]:
# Optimal plan over the year: solar vs wind generation by month (demand = annual/12 per month)
if result.get("monthly_balance") is not None:
    print("Monthly generation and balance (solar + wind over the year):")
    result["monthly_balance"]
else:
    print("Monthly breakdown available only when using flux_source='last_year_monthly'.")

Monthly generation and balance (solar + wind over the year):


## Tariff recommendation (with scraping)

Combine **scraped tariff data** (from the scraper) with the **optimisation** to recommend a tariff: we run optimisation once for your location and demand, then score each tariff by total cost (capex + grid import cost − export revenue + standing charge) over 5 years. Use either **example tariff data** below or pass a list of scraped `Tariff` objects from `ScrapeTariff.scrape()`.

In [33]:
# Tariff recommendation: recommend a tariff based on user inputs + (optional) scraped tariffs
from src.models.tariff_recommendation import recommend_tariff

# Example tariff data (or use scraped tariffs from ScrapeTariff.scrape() — same location/demand used)
# Unit rate in p/kWh, standing charge in p/day. With real scraping you get these from comparison sites.
example_tariffs = [
    {"new_supplier_name": "Octopus", "tariff_name": "Flexible", "unit_rate": 24.5, "standing_charge_day": 55.0, "is_green": True},
    {"new_supplier_name": "British Gas", "tariff_name": "Standard", "unit_rate": 28.2, "standing_charge_day": 60.0, "is_green": False},
    {"new_supplier_name": "EDF", "tariff_name": "Standard", "unit_rate": 26.8, "standing_charge_day": 52.0, "is_green": True},
    {"new_supplier_name": "Ovo", "tariff_name": "Better", "unit_rate": 25.0, "standing_charge_day": 58.0, "is_green": True},
]

# User inputs: location and demand (or from first scraped tariff: tariff.latitude, .longitude, .annual_electricity_kwh)
rec = recommend_tariff(
    example_tariffs,
    latitude,
    longitude,
    annual_consumption_kwh,
    solar_type,
    wind_type,
    export_price_per_kwh=0.05,
    optimize_over_years=5.0,
    prefer_green=True,
)

if rec.get("error"):
    print("Error:", rec["error"])
else:
    opt = rec["optimisation_result"]
    best = rec["recommended_tariff"]
    print("--- Tariff recommendation (with optimal solar/wind) ---")
    print(f"Optimal system: solar {opt['optimal_solar_kw']} kW, wind {opt['optimal_wind_kw']} kW")
    print(f"Annual import: {opt['annual_import_kwh']:.0f} kWh  |  export: {opt['annual_export_kwh']:.0f} kWh  |  capex: £{opt['capex']:.0f}")
    print()
    print(f"Recommended: {best.get('supplier_name')} - {best.get('tariff_name')}  (green: {best.get('is_green', False)})")
    print(f"  Unit rate: {best.get('unit_rate_p_per_kwh', 0):.1f}p/kWh  |  Standing charge: {best.get('standing_charge_p_per_day', 0):.1f}p/day")
    print(f"  Total cost over {rec['optimize_over_years']} years: £{rec['ranking'][0]['total_cost_gbp']:.2f}")
    print()
    print("Ranking (total cost = capex + grid cost − export over 5 years):")
    for r in rec["ranking"]:
        t = r["tariff"]
        print(f"  {r['rank']}. {t.get('supplier_name')} - {t.get('tariff_name')}: £{r['total_cost_gbp']:.2f}  (opex/yr: £{r['opex_per_year_gbp']:.2f})")
rec

--- Tariff recommendation (with optimal solar/wind) ---
Optimal system: solar 1.5 kW, wind 0.5 kW
Annual import: 1661 kWh  |  export: 0 kWh  |  capex: £3500

Recommended: Octopus - Flexible  (green: True)
  Unit rate: 24.5p/kWh  |  Standing charge: 55.0p/day
  Total cost over 5.0 years: £6538.11

Ranking (total cost = capex + grid cost − export over 5 years):
  1. Octopus - Flexible: £6538.11  (opex/yr: £607.62)
  2. Ovo - Better: £6634.38  (opex/yr: £626.88)
  3. EDF - Standard: £6674.34  (opex/yr: £634.87)
  4. British Gas - Standard: £6936.59  (opex/yr: £687.32)


{'optimisation_result': {'optimal_solar_kw': 1.5,
  'optimal_wind_kw': 0.5,
  'total_capacity_kw': 2.0,
  'annual_demand_kwh': 3500.0,
  'annual_generation_kwh': 1839.3,
  'annual_solar_generation_kwh': 1405.9,
  'annual_wind_generation_kwh': 433.4,
  'annual_import_kwh': 1660.7,
  'annual_export_kwh': 0.0,
  'demand_met_from_generation_pct': 52.6,
  'capex': 3500.0,
  'solar_capex': 2250.0,
  'wind_capex': 1250.0,
  'annual_net_opex': 406.86,
  'payback_solar_years': 6.5,
  'payback_wind_years': 11.8,
  'financials_0_year': {'capex': 3500.0, 'opex_total': 0.0, 'total': 3500.0},
  'financials_5_year': {'capex': 3500.0,
   'opex_total': 2034.31,
   'total': 5534.31},
  'financials_10_year': {'capex': 3500.0,
   'opex_total': 4068.62,
   'total': 7568.62},
  'monthly_balance':    month  solar_kwh  wind_kwh  total_gen_kwh  demand_kwh  import_kwh  \
  0    Jan       32.3      37.8           70.0       291.7       221.7   
  1    Feb       42.9      41.8           84.7       291.7       207

In [34]:
# Optional: daily flux from forecast (no generation) — for comparison with monthly historical
flux_daily = get_flux_daily(latitude, longitude)
flux_daily

,ghi_mj_per_m2,wind_speed_10m_max
date,,
2026-03-04 00:00:00+00:00,12.39,4.642467
2026-03-05 00:00:00+00:00,11.85,4.348850
2026-03-06 00:00:00+00:00,3.88,5.158004
2026-03-07 00:00:00+00:00,3.59,2.773536
2026-03-08 00:00:00+00:00,4.87,3.063495
2026-03-09 00:00:00+00:00,5.70,3.422353
2026-03-10 00:00:00+00:00,8.50,8.031967


In [35]:
# Switch generation types: SOLAR_TIERS["mid"], WIND_TIERS["premium"], etc.
# get_generation(..., SOLAR_TIERS["mid"], WIND_TIERS["premium"])
# get_optimised_system(..., flux_source="forecast")  # use 7-day forecast instead of last year